# LushProtein — Solution 1 Standalone (Raw Data → Decile & CRM Tiers)

**Fully self-contained.** Requires only the five raw data folders at project root.

**Profit formula (founder-scoped):** `Profit = Net Revenue − COGS` (no fulfillment/refund terms).

**Pipeline:**
1. Install dependencies
2. Load & merge raw Shopify transactions
3. Apply DQ + LP founder filters → finals cohort
4. Enrich with COGS from product master
5. Reproduce Solution 1: 5-tier profit & frequency deciles, CRM tiers, charts

References: `EDA/lushprotein_decile.ipynb`, `EDA/lushprotein_decile_v2.ipynb`


In [1]:
# ── 0. Install dependencies (run this cell first) ───────────────────────────
import importlib.util
import subprocess
import sys

REQUIRED = [
    "pandas", "numpy", "matplotlib", "seaborn", "scikit-learn",
    "openpyxl", "pyarrow",
]

missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All dependencies already installed.")


Installing: scikit-learn


In [2]:
import warnings
warnings.filterwarnings("ignore")

import json
import os
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    display = print
import matplotlib.ticker as mticker

# ── Project paths ─────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "1.customer_transaction").exists():
    alt = PROJECT_ROOT.parent
    if (alt / "1.customer_transaction").exists():
        PROJECT_ROOT = alt
    else:
        raise FileNotFoundError(
            "Open this notebook from the project root (folder containing 1.customer_transaction/)."
        )
os.chdir(PROJECT_ROOT)

RAW_ORDERS_DIR = PROJECT_ROOT / "1.customer_transaction"
STANDALONE_OUT = PROJECT_ROOT / "standalone_outputs"
STANDALONE_OUT.mkdir(exist_ok=True)

def _glob_one(folder: Path, pattern: str) -> Path:
    matches = sorted(folder.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No file matching {pattern} in {folder}")
    return matches[0]

ORDER_FILES = sorted(RAW_ORDERS_DIR.glob("1_*.xlsx"))
PRODUCTS_FILE = _glob_one(PROJECT_ROOT / "2.product_master", "*.xlsx")
DISCOUNTS_FILE = _glob_one(PROJECT_ROOT / "3.Discounts", "*.csv")
CAMPAIGNS_FILE = _glob_one(PROJECT_ROOT / "4.Campaigns", "*.csv")
RECHARGE_DIR = PROJECT_ROOT / "5.Recharge_data"

EXCLUDE_HANDLE = "better-whey-protein-elite"
EXCLUDE_MONTHS = {7, 11}
ANALYSIS_START = pd.Timestamp("2022-01-01", tz="Asia/Singapore")
ANALYSIS_DATE = pd.Timestamp("2026-04-30", tz="UTC")
MARGIN_PROXY = 0.40
N_TIERS = 5
DECILE_CHART_ORDER = [f"D{i}" for i in range(10, 0, -1)]

FX_RATES_TO_SGD = {"SG": 1.0, "MY": 1.0 / 3.30, "HK": 1.0 / 6.10}
PRODUCT_MAP = {
    "lean-protein": "Lean Protein", "lean_protein": "Lean Protein",
    "clear-protein": "Clear Protein", "clear_protein": "Clear Protein",
    "collagen": "Collagen Glow", "soy-protein": "Soy Protein",
    "protein-bar": "Protein Bar", "shaker": "Accessories", "starter-kit": "Accessories",
}
MARKETPLACE_KEYWORDS = ["shopee", "lazada", "tokopedia", "redmart", "grab"]
CATEGORIES = [
    "Clear Protein", "Lean Protein", "Collagen Glow",
    "Accessories", "Soy Protein", "Other", "Unknown",
]

print("Project root:", PROJECT_ROOT)
print("Order files:", len(ORDER_FILES))
print("Products:", PRODUCTS_FILE.name)
print("Output dir:", STANDALONE_OUT)



Project root: c:\Users\adity\Documents\Aditya SMU\SMU Sem 5\Lush Protein SMU X\LushProtein_Project_Data_20260505
Order files: 7
Products: 2_1.products_master_20260505.xlsx
Output dir: c:\Users\adity\Documents\Aditya SMU\SMU Sem 5\Lush Protein SMU X\LushProtein_Project_Data_20260505\standalone_outputs


In [3]:
# ── Helper functions (mirrors EDA/00_config.py + aditya_findings/_shared.py) ─

def classify_product(handle) -> str:
    if pd.isna(handle):
        return "Unknown"
    h = str(handle).lower()
    for kw, label in PRODUCT_MAP.items():
        if kw in h:
            return label
    return "Other"


def classify_channel(row) -> str:
    tags = str(row.get("Tags", "") or "").lower()
    utm = str(row.get("Browser: UTM Source", "") or "").lower()
    name = str(row.get("Name", "") or "").lower()
    if any(k in tags for k in MARKETPLACE_KEYWORDS):
        return "Marketplace"
    if "subscription" in tags or "yotpo subscriptions" in tags or "lpsg" in name[:4]:
        return "Subscription"
    if utm in ("facebook", "instagram", "tiktok"):
        return "Paid Social"
    if utm in ("google", "bing"):
        return "Paid Search"
    if utm == "affiliate":
        return "Affiliate"
    if utm in ("shopify_email", "email", "klaviyo"):
        return "Email"
    return "Direct / Organic"


def store_prefix(name: str) -> str:
    if pd.isna(name):
        return "Unknown"
    n = str(name).upper().replace("#", "")
    if n.startswith("LPMY"):
        return "MY"
    if n.startswith("LPHK"):
        return "HK"
    if n.startswith("LPSG") or n.startswith("LP"):
        return "SG"
    return "Other"


def assign_decile(series: pd.Series) -> pd.Series:
    ranks = series.rank(method="first", ascending=True)
    return pd.qcut(ranks, q=10, labels=DECILE_CHART_ORDER)


def crm_tier(row) -> str:
    if row.get("is_top_both"):
        return "VIP"
    if row.get("is_top_profit") or row.get("profit_decile_true") == "D1":
        return "Profit_D1"
    if row.get("is_top_freq") or row.get("freq_decile_true") == "D1":
        return "Freq_D1"
    return "Standard"


def _safe_str(val, default="") -> str:
    """Convert cell values to str without boolean checks on pd.NA."""
    if val is None:
        return default
    try:
        if pd.isna(val):
            return default
    except (TypeError, ValueError):
        pass
    s = str(val).strip()
    if s in ("nan", "None", "<NA>", ""):
        return default
    return s


def sku_label(row) -> str:
    handle = _safe_str(row.get("Line: Product Handle"), "unknown")
    if handle == "unknown":
        handle = _safe_str(row.get("Line: SKU"), "unknown")
    variant = _safe_str(row.get("Line: Variant Title"), "")
    if "/" in variant:
        flavour = variant.split("/")[-1].strip()
    else:
        flavour = variant[:30] if variant else "default"
    return f"{handle}|{flavour}"[:80]


def clean_object_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.select_dtypes(include="object").columns:
        out[col] = out[col].where(out[col].notna(), other=pd.NA)
        out[col] = out[col].apply(lambda x: str(x) if pd.notna(x) else pd.NA)
    return out


def build_cogs_map(products_df: pd.DataFrame) -> dict[str, float]:
    """COGS from product master Cost per item (Variant SKU key)."""
    cost_map: dict[str, float] = {}
    sku_col = "Variant SKU" if "Variant SKU" in products_df.columns else "SKU"
    cost_col = "Cost per item" if "Cost per item" in products_df.columns else None
    if cost_col is None:
        return cost_map
    for _, r in products_df[[sku_col, cost_col]].dropna(subset=[cost_col]).iterrows():
        cost_map[str(r[sku_col]).strip()] = float(r[cost_col])
    return cost_map


print("Helpers loaded.")

def assign_decile_5(series: pd.Series, n_tiers: int = N_TIERS) -> pd.Series:
    """D1 = best. Five tiers for Solution 1 (founder request)."""
    ranks = series.rank(method="first", ascending=True)
    return pd.qcut(ranks, q=n_tiers, labels=[f"D{i}" for i in range(n_tiers, 0, -1)])




Helpers loaded.


## Part A — Load raw data

In [4]:
# ── A1. Shopify order transactions ────────────────────────────────────────────
assert ORDER_FILES, f"No 1_*.xlsx files in {RAW_ORDERS_DIR}"

raw_chunks = []
for f in ORDER_FILES:
    print(f"  {f.name} ...", end=" ")
    df = pd.read_excel(f, dtype={"ID": str, "Customer: ID": str})
    print(f"{len(df):,} rows")
    raw_chunks.append(df)

raw = pd.concat(raw_chunks, ignore_index=True)
raw["Processed At"] = pd.to_datetime(raw["Processed At"], utc=True, errors="coerce")
raw["order_date"] = raw["Processed At"].dt.tz_convert("Asia/Singapore").dt.normalize()
raw["store"] = raw["Name"].apply(store_prefix)

orders_cols = [
    "ID", "Name", "Tags", "order_date", "store", "Customer: ID", "Currency",
    "Price: Total", "Price: Total Discount", "Price: Total Shipping",
    "Payment: Status", "Order Fulfillment Status",
    "Shipping: Country", "Browser: UTM Source", "Browser: UTM Medium",
    "Line: Product Handle", "Line: Title", "Line: Variant Title", "Line: SKU",
    "Line: Price", "Line: Quantity",
]
existing_cols = [c for c in orders_cols if c in raw.columns]
orders_df = raw[raw["Top Row"] == 1][existing_cols].copy()
orders_df = orders_df.rename(columns={"ID": "order_id", "Customer: ID": "customer_id"})
orders_df = orders_df.dropna(subset=["customer_id", "order_date"])
if "Payment: Status" in orders_df.columns:
    orders_df = orders_df[
        orders_df["Payment: Status"].isin(["paid", "partially_refunded"]) | orders_df["Payment: Status"].isna()
    ]
orders_df = orders_df[orders_df["Order Fulfillment Status"].fillna("") != "restocked"]

orders_df["_fx"] = orders_df["store"].map(FX_RATES_TO_SGD).fillna(1.0)
for col in ["Price: Total", "Price: Total Discount", "Price: Total Shipping", "Line: Price"]:
    if col in orders_df.columns:
        orders_df[col] = pd.to_numeric(orders_df[col], errors="coerce").fillna(0) * orders_df["_fx"]
orders_df.drop(columns=["_fx"], inplace=True)
orders_df["Currency"] = "SGD"
orders_df["channel"] = orders_df.apply(classify_channel, axis=1)
orders_df["product_category"] = orders_df["Line: Product Handle"].apply(classify_product)
orders_df["has_discount"] = orders_df["Price: Total Discount"].fillna(0) > 0
orders_df["is_subscription"] = orders_df["Tags"].fillna("").str.lower().str.contains("subscription|yotpo subscriptions")

line_cols = [
    "ID", "Customer: ID", "order_date", "store",
    "Line: Product Handle", "Line: Title", "Line: Variant Title",
    "Line: SKU", "Line: Quantity", "Line: Price", "Line: Discount", "Line: Total",
]
lines_df = raw[raw["Line: Type"] == "Line Item"][[c for c in line_cols if c in raw.columns]].copy()
lines_df = lines_df.rename(columns={"ID": "order_id", "Customer: ID": "customer_id"})
lines_df = lines_df.dropna(subset=["customer_id", "order_date"])
lines_df["product_category"] = lines_df["Line: Product Handle"].apply(classify_product)
lines_df["_fx"] = lines_df["store"].map(FX_RATES_TO_SGD).fillna(1.0)
for col in ["Line: Price", "Line: Discount", "Line: Total"]:
    if col in lines_df.columns:
        lines_df[col] = pd.to_numeric(lines_df[col], errors="coerce").fillna(0) * lines_df["_fx"]
lines_df.drop(columns=["_fx"], inplace=True)

cust_base = (
    orders_df.sort_values("order_date")
    .groupby("customer_id")
    .agg(
        first_order_date=("order_date", "min"),
        last_order_date=("order_date", "max"),
        total_orders=("order_id", "count"),
        total_revenue=("Price: Total", "sum"),
        total_discount=("Price: Total Discount", "sum"),
        ever_subscribed=("is_subscription", "any"),
        ever_discounted=("has_discount", "any"),
        first_channel=("channel", "first"),
        first_product_cat=("product_category", "first"),
        first_store=("store", "first"),
    )
    .reset_index()
)
cust_base["cohort_month"] = cust_base["first_order_date"].dt.to_period("M")
second_orders = (
    orders_df.sort_values("order_date").groupby("customer_id", as_index=False).nth(1)[["customer_id", "order_date"]]
    .rename(columns={"order_date": "second_order_date"})
)
cust_base = cust_base.merge(second_orders, on="customer_id", how="left")
cust_base["days_to_second"] = (cust_base["second_order_date"] - cust_base["first_order_date"]).dt.days
cust_base["is_repeat"] = cust_base["total_orders"] >= 2
cust_base["lifespan_days"] = (cust_base["last_order_date"] - cust_base["first_order_date"]).dt.days
cust_base["recency_days"] = (ANALYSIS_DATE - cust_base["last_order_date"]).dt.days

orders_df = clean_object_cols(orders_df)
lines_df = clean_object_cols(lines_df)
cust_base = clean_object_cols(cust_base)

print(f"\nOrders: {len(orders_df):,} | Lines: {len(lines_df):,} | Customers: {len(cust_base):,}")
print(f"Date range: {orders_df['order_date'].min().date()} → {orders_df['order_date'].max().date()}")

# Aliases for EDA comparisons (raw snapshot before finals filters)
orders_all = orders_df.copy()
lines_all = lines_df.copy()
cust_all = cust_base.copy()




  1_1.orders-2020_20260505.xlsx ... 12,201 rows
  1_2.orders-2021_20260505.xlsx ... 33,196 rows
  1_3.orders-2022_20260505.xlsx ... 18,563 rows
  1_4.orders-2023_20260505.xlsx ... 12,962 rows
  1_5.orders-2024_20260505.xlsx ... 26,373 rows
  1_6.orders-2025_20260505.xlsx ... 40,932 rows
  1_7.orders-2026_20260505.xlsx ... 9,601 rows

Orders: 27,350 | Lines: 50,963 | Customers: 13,780
Date range: 2020-01-01 → 2026-03-31


In [5]:
# ── A2. Ancillary raw tables ──────────────────────────────────────────────────
products_df = pd.read_excel(PRODUCTS_FILE)
discounts_df = pd.read_csv(DISCOUNTS_FILE, encoding="utf-8", encoding_errors="replace")
campaigns_df = pd.read_csv(CAMPAIGNS_FILE, encoding="utf-8", encoding_errors="replace")

rc_orders = pd.read_excel(_glob_one(RECHARGE_DIR, "*orders_combined*.xlsx"))
rc_checkout = pd.read_excel(_glob_one(RECHARGE_DIR, "*checkout*.xlsx"))
rc_reactivated = pd.read_excel(_glob_one(RECHARGE_DIR, "*reactivated*.xlsx"))
rc_churned = pd.read_excel(_glob_one(RECHARGE_DIR, "*churned*.xlsx"))
rc_recurring = pd.read_excel(_glob_one(RECHARGE_DIR, "*recurring*.xlsx"))

cost_map = build_cogs_map(products_df)
print("Products:", len(products_df), "| Discounts:", len(discounts_df), "| Campaigns:", len(campaigns_df))
print("Recharge tables:", len(rc_orders), len(rc_checkout), len(rc_reactivated), len(rc_churned), len(rc_recurring))
print("SKUs with COGS from product master:", len(cost_map))


Products: 167 | Discounts: 367 | Campaigns: 137033
Recharge tables: 1215 1094 50 526 650
SKUs with COGS from product master: 53


## Part B — Build finals cohort (DQ + LP filters)

Mirrors `EDA/13_build_finals_datasets.py`:
- **Layer 1:** DQ-02/03/04 order drops
- **Layer 2:** LP-F01/F02/F03/F04 customer flags
- **Layer 0:** 2022+ order window
- **Layer 3:** Drop Jul/Nov order months + elite SKU lines


In [6]:
def rebuild_customers(orders_df, lines_df, cust_seed, analysis_date=ANALYSIS_DATE):
    orders_df = orders_df.copy()
    orders_df["_rev_sgd"] = pd.to_numeric(orders_df["Price: Total"], errors="coerce").fillna(0)
    agg = (
        orders_df.groupby("customer_id")
        .agg(total_orders=("order_id", "count"), total_revenue=("_rev_sgd", "sum"), last_order_date=("order_date", "max"))
        .reset_index()
    )
    orders_df.drop(columns=["_rev_sgd"], inplace=True, errors="ignore")

    active_ids = set(agg["customer_id"])
    cust = cust_seed[cust_seed["customer_id"].isin(active_ids)].copy()
    drop_cols = [
        "total_orders", "total_revenue", "first_order_date", "last_order_date", "is_repeat",
        "acq_year", "acq_month", "lifespan_days", "recency_days", "days_to_second", "second_order_date",
        "first_disc_depth", "first_disc_bin", "first_order_source", "first_order_pos",
        "exclude_elite_buyer", "exclude_51pct", "exclude_promo_month", "finals_eligible",
    ]
    cust = cust.drop(columns=[c for c in drop_cols if c in cust.columns])
    cust = cust.merge(agg, on="customer_id", how="inner")

    acq_cols = cust_seed[["customer_id", "first_order_date"]].drop_duplicates("customer_id")
    cust = cust.merge(acq_cols, on="customer_id", how="left")

    cust["is_repeat"] = cust["total_orders"] >= 2
    cust["acq_year"] = cust["first_order_date"].dt.year
    cust["acq_month"] = cust["first_order_date"].dt.month
    cust["lifespan_days"] = (cust["last_order_date"] - cust["first_order_date"]).dt.days
    cust["recency_days"] = (analysis_date - cust["last_order_date"]).dt.days

    first_ord = orders_df.sort_values("order_date").groupby("customer_id").first().reset_index()
    first_ord["first_rev"] = pd.to_numeric(first_ord["Price: Total"], errors="coerce").fillna(0)
    first_ord["first_disc"] = pd.to_numeric(first_ord["Price: Total Discount"], errors="coerce").fillna(0)
    first_ord["first_disc_depth"] = np.where(
        (first_ord["first_rev"] + first_ord["first_disc"]) > 0,
        first_ord["first_disc"] / (first_ord["first_rev"] + first_ord["first_disc"]), 0,
    )
    first_ord["first_disc_bin"] = pd.cut(
        first_ord["first_disc_depth"],
        bins=[-0.001, 0.001, 0.05, 0.10, 0.20, 0.30, 0.50, 1.01],
        labels=["0%", "1-5%", "6-10%", "11-20%", "21-30%", "31-50%", "51%+"],
    )
    if "Source" in first_ord.columns:
        first_ord["first_order_source"] = first_ord["Source"].fillna("unknown").str.lower()
    else:
        first_ord["first_order_source"] = "unknown"

    second_ord = orders_df.sort_values("order_date").groupby("customer_id", as_index=False).nth(1)[["customer_id", "order_date"]]
    second_ord = second_ord.rename(columns={"order_date": "second_order_date"})
    cust = cust.merge(first_ord[["customer_id", "first_disc_depth", "first_disc_bin", "first_order_source"]], on="customer_id", how="left")
    cust = cust.merge(second_ord, on="customer_id", how="left")
    cust["days_to_second"] = (cust["second_order_date"] - cust["first_order_date"]).dt.days
    cust["first_order_pos"] = cust["first_order_source"] == "pos"

    for col in ["first_channel", "ever_subscribed", "ever_discounted", "first_product_cat"]:
        if col not in cust.columns and col in cust_seed.columns:
            cust = cust.merge(cust_seed[["customer_id", col]].drop_duplicates("customer_id"), on="customer_id", how="left")

    elite_customers = set(
        lines_df[lines_df["Line: Product Handle"].fillna("").str.contains(EXCLUDE_HANDLE, case=False)]["customer_id"]
    )
    cust["exclude_elite_buyer"] = cust["customer_id"].isin(elite_customers)
    cust["exclude_51pct"] = cust["first_disc_bin"].astype(str) == "51%+"
    cust["exclude_promo_month"] = cust["acq_month"].isin(EXCLUDE_MONTHS)
    cust["finals_eligible"] = (
        (cust["first_order_date"] >= ANALYSIS_START)
        & ~cust["exclude_elite_buyer"]
        & ~cust["exclude_51pct"]
        & ~cust["exclude_promo_month"]
    )
    return cust


# Enrich orders with POS/web Source from raw files
src_chunks = []
for f in ORDER_FILES:
    if f.stat().st_size < 1000:
        continue
    src = pd.read_excel(f, usecols=["ID", "Top Row", "Source"], dtype=str, engine="openpyxl")
    src = src[src["Top Row"] == "1"].rename(columns={"ID": "order_id"})
    src_chunks.append(src[["order_id", "Source"]])
if src_chunks:
    src_df = pd.concat(src_chunks, ignore_index=True).drop_duplicates("order_id")
    src_df["order_id"] = src_df["order_id"].astype(str)
    orders_df["order_id"] = orders_df["order_id"].astype(str)
    orders_df = orders_df.merge(src_df, on="order_id", how="left")

orders_df["order_date"] = pd.to_datetime(orders_df["order_date"], utc=True)
lines_df["order_date"] = pd.to_datetime(lines_df["order_date"], utc=True)
orders_df["order_id"] = orders_df["order_id"].astype(str)
lines_df["order_id"] = lines_df["order_id"].astype(str)
cust_base["first_order_date"] = pd.to_datetime(cust_base["first_order_date"], utc=True)

# Layer 1 — DQ
_rev = pd.to_numeric(orders_df["Price: Total"], errors="coerce").fillna(0)
_disc = pd.to_numeric(orders_df["Price: Total Discount"], errors="coerce").fillna(0)
_dq02 = (_rev == 0) & (_disc == 0)
_dq03 = (_rev == 0) & (_disc > 0)
_dq04 = orders_df["Tags"].fillna("").str.lower().str.contains("wholesale") | (_rev > 5000)
orders_dq = orders_df[~(_dq02 | _dq03 | _dq04)].copy()
lines_dq = lines_df[lines_df["order_id"].isin(set(orders_dq["order_id"]))].copy()
cust_dq = rebuild_customers(orders_dq, lines_dq, cust_base)

# Layer 2
finals_ids = set(cust_dq[cust_dq["finals_eligible"]]["customer_id"])
orders_l2 = orders_dq[orders_dq["customer_id"].isin(finals_ids)].copy()
lines_l2 = lines_dq[lines_dq["customer_id"].isin(finals_ids)].copy()
cust_l2 = cust_dq[cust_dq["finals_eligible"]].copy()

# Layer 0
orders_l2 = orders_l2[orders_l2["order_date"] >= ANALYSIS_START].copy()
lines_l2 = lines_l2[lines_l2["order_id"].isin(set(orders_l2["order_id"]))].copy()

# Layer 3
_jul_nov = orders_l2["order_date"].dt.month.isin(EXCLUDE_MONTHS)
orders_finals = orders_l2[~_jul_nov].copy()
lines_finals = lines_l2[
    lines_l2["order_id"].isin(set(orders_finals["order_id"]))
    & ~lines_l2["Line: Product Handle"].fillna("").str.contains(EXCLUDE_HANDLE, case=False)
].copy()

customers = rebuild_customers(orders_finals, lines_finals, cust_l2)
customers = customers[customers["customer_id"].isin(finals_ids)].copy()
customers["finals_eligible"] = True
orders = orders_finals.copy()
lines = lines_finals.copy()

print("Finals cohort:")
print(f"  Orders:    {len(orders):,}")
print(f"  Lines:     {len(lines):,}")
print(f"  Customers: {len(customers):,}")
print(f"  Finals-eligible flag: {customers['finals_eligible'].sum():,}")


Finals cohort:
  Orders:    8,955
  Lines:     14,448
  Customers: 5,694
  Finals-eligible flag: 5,694


## Part C — Margin enrichment (COGS from product master)

In [7]:
OUT_DIR = STANDALONE_OUT / "solution1"
OUT_DIR.mkdir(exist_ok=True)

# Attach unit costs and gross profit to line items
lines["sku_key"] = lines["Line: SKU"].astype(str).str.strip()
lines["unit_cost"] = lines["sku_key"].map(cost_map)
lines["line_rev"] = pd.to_numeric(lines["Line: Total"], errors="coerce").fillna(0)
lines["qty"] = pd.to_numeric(lines["Line: Quantity"], errors="coerce").fillna(1).clip(lower=1)
lines["cogs"] = lines["unit_cost"] * lines["qty"]
lines["gross_profit"] = np.where(lines["unit_cost"].notna(), lines["line_rev"] - lines["cogs"], lines["line_rev"] * MARGIN_PROXY)
lines["has_cogs"] = lines["unit_cost"].notna()
lines["margin_pct"] = np.where(lines["line_rev"] > 0, lines["gross_profit"] / lines["line_rev"], np.nan)

order_gp = lines.groupby("order_id").agg(
    order_gp=("gross_profit", "sum"), order_cogs=("cogs", "sum"), order_rev=("line_rev", "sum"),
    n_categories=("product_category", "nunique"),
).reset_index()
order_gp["order_margin_pct"] = np.where(order_gp["order_rev"] > 0, order_gp["order_gp"] / order_gp["order_rev"], np.nan)
orders = orders.merge(order_gp, on="order_id", how="left")
orders["order_gp"] = orders["order_gp"].fillna(pd.to_numeric(orders["Price: Total"], errors="coerce").fillna(0) * MARGIN_PROXY)

cat_ever = lines.groupby("customer_id")["product_category"].nunique().reset_index(name="n_categories_ever")
cust_rev = orders.groupby("customer_id").agg(
    finals_revenue=("Price: Total", lambda s: pd.to_numeric(s, errors="coerce").sum()),
    finals_orders=("order_id", "count"),
    true_gross_profit=("order_gp", "sum"),
).reset_index()
customers = customers.merge(cat_ever, on="customer_id", how="left").merge(cust_rev, on="customer_id", how="left")
customers["true_gross_profit"] = customers["true_gross_profit"].fillna(customers["total_revenue"] * MARGIN_PROXY)
customers["n_categories_ever"] = customers["n_categories_ever"].fillna(1).astype(int)

covered = lines[lines["has_cogs"]]
cust_gp = covered.groupby("customer_id").agg(
    true_gp_covered=("gross_profit", "sum"), rev_covered=("line_rev", "sum"),
).reset_index()
rev_all = lines.groupby("customer_id")["line_rev"].sum().reset_index(name="rev_total_lines")
cust_gp = rev_all.merge(cust_gp, on="customer_id", how="left")
cust_gp["cogs_coverage_pct"] = np.where(
    cust_gp["rev_total_lines"] > 0, cust_gp["rev_covered"].fillna(0) / cust_gp["rev_total_lines"], 0,
)
customers = customers.merge(cust_gp[["customer_id", "cogs_coverage_pct"]], on="customer_id", how="left")
customers["cogs_coverage_pct"] = customers["cogs_coverage_pct"].fillna(0)
customers["avg_margin_pct"] = np.where(
    customers["finals_revenue"] > 0, customers["true_gross_profit"] / customers["finals_revenue"], np.nan,
)

pool = customers[customers["finals_eligible"]].copy()
pool["profit_decile_true"] = assign_decile(pool["true_gross_profit"])
pool["freq_decile_true"] = assign_decile(pool["finals_orders"].fillna(pool["total_orders"]))
pool["is_top_profit"] = pool["profit_decile_true"] == "D1"
pool["is_top_freq"] = pool["freq_decile_true"] == "D1"
pool["is_top_both"] = pool["is_top_profit"] & pool["is_top_freq"]
pool["crm_tier"] = pool.apply(crm_tier, axis=1)
customers = customers.merge(
    pool[["customer_id", "profit_decile_true", "freq_decile_true", "is_top_profit", "is_top_freq", "is_top_both", "crm_tier"]],
    on="customer_id", how="left",
)

# Save standalone parquet cache (optional — for inspection)
for name, df in [("orders", orders), ("lines", lines), ("customers", customers)]:
    df.to_parquet(OUT_DIR / f"{name}.parquet", index=False)

manifest = {
    "source": "solution1_from_raw.ipynb",
    "row_counts": {"orders": len(orders), "lines": len(lines), "customers": len(customers)},
    "pct_lines_with_cogs": round(lines["has_cogs"].mean() * 100, 1),
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))



{
  "source": "solution2_standalone_from_raw.ipynb",
  "row_counts": {
    "orders": 8955,
    "lines": 14448,
    "customers": 5694
  },
  "pct_lines_with_cogs": 21.0
}


## Part D — Marketplace exclusion & decile pool


In [ ]:
# Channel mix — flag 100% Marketplace accounts (Shopee/Lazada aggregator pattern)
cust = customers.copy()
orders = orders.copy()

cust_channel_mix = orders.groupby("customer_id")["channel"].value_counts(normalize=True).unstack(fill_value=0)
mkt_col = cust_channel_mix.get("Marketplace", pd.Series(0, index=cust_channel_mix.index))
all_marketplace_ids = set(cust_channel_mix[mkt_col == 1.0].index.astype(str))

cust["is_all_marketplace"] = cust["customer_id"].astype(str).isin(all_marketplace_ids)

print("Orders by channel (finals cohort):")
print(orders["channel"].value_counts().to_string())
print(f"\nAll-Marketplace customers to exclude: {len(all_marketplace_ids):,}")

# Decile pool: finals_eligible AND not all-marketplace
pool_mask = (cust["finals_eligible"] == True) & (~cust["is_all_marketplace"])
df = cust[pool_mask].copy()

print(f"\nFilter funnel:")
print(f"  Finals customers          : {len(cust):,}")
print(f"  - all-marketplace excluded: {len(df):,}")
print(f"  Final decile pool         : {len(df):,}")
print(f"\nProfit range: S${df['true_gross_profit'].min():,.2f} – S${df['true_gross_profit'].max():,.2f}")
print(f"Order count range: {df['finals_orders'].min()} – {df['finals_orders'].max()}")


## Part E — Two 5-tier deciles (Profit & Order Frequency)


In [ ]:
df["profit_decile"] = assign_decile_5(df["true_gross_profit"])
df["order_freq_decile"] = assign_decile_5(df["finals_orders"])

print("Profit decile counts:")
print(df["profit_decile"].value_counts().sort_index().to_string())
print("\nOrder frequency decile counts:")
print(df["order_freq_decile"].value_counts().sort_index().to_string())

profit_summary = (
    df.groupby("profit_decile", observed=False)
    .agg(
        n_customers=("customer_id", "count"),
        total_profit=("true_gross_profit", "sum"),
        avg_profit=("true_gross_profit", "mean"),
        median_profit=("true_gross_profit", "median"),
        avg_revenue=("finals_revenue", "mean"),
        avg_orders=("finals_orders", "mean"),
        avg_margin_pct=("avg_margin_pct", "mean"),
    )
    .reset_index()
)
total_profit_pool = profit_summary["total_profit"].sum()
profit_summary["pct_of_total_profit"] = profit_summary["total_profit"] / total_profit_pool
profit_summary["cum_pct_profit"] = profit_summary["pct_of_total_profit"].cumsum()

freq_summary = (
    df.groupby("order_freq_decile", observed=False)
    .agg(
        n_customers=("customer_id", "count"),
        total_orders=("finals_orders", "sum"),
        avg_orders=("finals_orders", "mean"),
        avg_profit=("true_gross_profit", "mean"),
    )
    .reset_index()
)
total_orders_pool = freq_summary["total_orders"].sum()
freq_summary["pct_of_total_orders"] = freq_summary["total_orders"] / total_orders_pool

print(f"\nDECILE BY PROFIT — pool={len(df):,} | total profit=S${total_profit_pool:,.0f}")
display(profit_summary)
d1_pct = profit_summary.loc[profit_summary["profit_decile"] == "D1", "pct_of_total_profit"].values[0]
print(f"Top 20% (D1) generate {d1_pct:.1%} of total profit")


In [ ]:
# Profit decile charts
OUT = STANDALONE_OUT / "solution1"
OUT.mkdir(exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
tiers = profit_summary["profit_decile"].astype(str)
colors = ["#2E86AB" if t == "D1" else "#A8DADC" for t in tiers]

axes[0].bar(tiers, profit_summary["pct_of_total_profit"] * 100, color=colors, edgecolor="white")
axes[0].set_title("% of Total Profit by Tier")
axes[0].set_ylabel("% of profit")
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter())

axes[1].bar(tiers, profit_summary["avg_profit"], color=colors, edgecolor="white")
axes[1].set_title("Avg Profit per Customer")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"S${x:,.0f}"))

axes[2].plot(range(1, N_TIERS + 1), profit_summary["cum_pct_profit"] * 100, marker="o", color="#2E86AB", lw=2)
axes[2].set_title("Cumulative Profit Concentration")
axes[2].set_xticks(range(1, N_TIERS + 1))
axes[2].set_xticklabels([f"D{i}" for i in range(1, N_TIERS + 1)])
axes[2].yaxis.set_major_formatter(mticker.PercentFormatter())
axes[2].axhline(80, color="gray", ls="--", alpha=0.5)

plt.tight_layout()
plt.savefig(OUT / "decile_profit.png", dpi=150)
plt.show()
print("Saved:", OUT / "decile_profit.png")


In [ ]:
# Order frequency decile charts
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
tiers_f = freq_summary["order_freq_decile"].astype(str)
colors_f = ["#E76F51" if t == "D1" else "#F4A261" for t in tiers_f]

axes[0].bar(tiers_f, freq_summary["avg_orders"], color=colors_f, edgecolor="white")
axes[0].set_title("Avg Orders per Customer")

axes[1].bar(tiers_f, freq_summary["pct_of_total_orders"] * 100, color=colors_f, edgecolor="white")
axes[1].set_title("% of Total Orders")
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())

axes[2].bar(tiers_f, freq_summary["avg_profit"], color=colors_f, edgecolor="white")
axes[2].set_title("Avg Profit by Frequency Tier")
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"S${x:,.0f}"))

plt.tight_layout()
plt.savefig(OUT / "decile_order_frequency.png", dpi=150)
plt.show()
print("Saved:", OUT / "decile_order_frequency.png")


## Part F — Overlap heatmap (Profit D1 × Frequency D1)


In [ ]:
d1_profit = set(df[df["profit_decile"] == "D1"]["customer_id"])
d1_freq = set(df[df["order_freq_decile"] == "D1"]["customer_id"])
d1_both = d1_profit & d1_freq

print(f"D1 profit only     : {len(d1_profit):,}")
print(f"D1 frequency only  : {len(d1_freq):,}")
print(f"D1 BOTH (overlap)  : {len(d1_both):,}")

cross = pd.crosstab(df["profit_decile"], df["order_freq_decile"])
all_tiers = [f"D{i}" for i in range(1, N_TIERS + 1)]
cross = cross.reindex(index=all_tiers, columns=all_tiers, fill_value=0)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cross.values, cmap="Blues")
ax.set_xticks(range(N_TIERS))
ax.set_yticks(range(N_TIERS))
ax.set_xticklabels(all_tiers)
ax.set_yticklabels(all_tiers)
ax.set_xlabel("Order Frequency Decile")
ax.set_ylabel("Profit Decile")
ax.set_title("Profit × Frequency Tier Overlap")
for i in range(N_TIERS):
    for j in range(N_TIERS):
        ax.text(j, i, f"{cross.values[i, j]:,}", ha="center", va="center", color="black", fontsize=9)
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(OUT / "decile_overlap_heatmap.png", dpi=150)
plt.show()
print("Saved:", OUT / "decile_overlap_heatmap.png")


## Part G — CRM treatment tiers (v2 logic)


In [ ]:
# Tier assignment — same priority as lushprotein_decile_v2.ipynb
d1_profit_ids = set(df[df["profit_decile"] == "D1"]["customer_id"])
d1_freq_ids = set(df[df["order_freq_decile"] == "D1"]["customer_id"])
tier1_ids = d1_profit_ids & d1_freq_ids
tier2_ids = d1_profit_ids - tier1_ids
tier3_ids = d1_freq_ids - tier1_ids

d2_profit_ids = set(df[df["profit_decile"] == "D2"]["customer_id"])
d2_freq_ids = set(df[df["order_freq_decile"] == "D2"]["customer_id"])
already_placed = tier1_ids | tier2_ids | tier3_ids
tier4_ids = (d2_profit_ids | d2_freq_ids) - already_placed

def assign_tier(cid):
    if cid in tier1_ids:
        return "Tier 1 (D1 Both)"
    if cid in tier2_ids:
        return "Tier 2 (D1 Profit Only)"
    if cid in tier3_ids:
        return "Tier 3 (D1 Frequency Only)"
    if cid in tier4_ids:
        return "Tier 4 (D2 Profit or Frequency)"
    return "Untiered"

df["customer_tier"] = df["customer_id"].apply(assign_tier)
df["is_top_profit"] = df["profit_decile"] == "D1"
df["is_top_freq"] = df["order_freq_decile"] == "D1"
df["is_top_both"] = df["is_top_profit"] & df["is_top_freq"]

print("Customers per CRM tier:")
print(df["customer_tier"].value_counts().to_string())

tier_order = [
    "Tier 1 (D1 Both)", "Tier 2 (D1 Profit Only)",
    "Tier 3 (D1 Frequency Only)", "Tier 4 (D2 Profit or Frequency)",
]
tier_margin = (
    df[df["customer_tier"] != "Untiered"]
    .groupby("customer_tier")
    .agg(
        n=("customer_id", "count"),
        avg_margin=("avg_margin_pct", "mean"),
        avg_gp=("true_gross_profit", "mean"),
        avg_rev=("finals_revenue", "mean"),
        avg_orders=("finals_orders", "mean"),
    )
    .reindex(tier_order)
)
display(tier_margin)


In [ ]:
# Reinvestment table (20% of contribution margin)
REINVEST_PCT = 0.20
tier_summary = (
    df[df["customer_tier"] != "Untiered"]
    .groupby("customer_tier")
    .agg(
        n_customers=("customer_id", "count"),
        total_gp=("true_gross_profit", "sum"),
        avg_gp=("true_gross_profit", "mean"),
    )
    .reindex(tier_order)
)
tier_summary["total_reinvestable"] = tier_summary["total_gp"] * REINVEST_PCT
tier_summary["reinvest_per_cust"] = tier_summary["avg_gp"] * REINVEST_PCT
print(f"Reinvestment at {REINVEST_PCT:.0%} of GP:")
display(tier_summary)
print(f"Total reinvestable: S${tier_summary['total_reinvestable'].sum():,.2f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(tier_summary.index.astype(str), tier_summary["total_reinvestable"], color="#457B9D", edgecolor="white")
ax.set_title("Reinvestment Budget by CRM Tier (20% of GP)")
ax.set_ylabel("S$")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.savefig(OUT / "tier_reinvestment.png", dpi=150)
plt.show()


## Part H — Behavioral profiles by tier


In [ ]:
# Recency from finals orders
most_recent = orders["order_date"].max()
if "last_order_date" not in df.columns:
    last_order = orders.groupby("customer_id")["order_date"].max().reset_index(name="last_order_date")
    df = df.merge(last_order, on="customer_id", how="left")
df["last_order_date"] = pd.to_datetime(df["last_order_date"], utc=True)
df["days_since_last_order"] = (most_recent - df["last_order_date"]).dt.days

print("=" * 70)
print("BEHAVIORAL PROFILE BY TIER")
print("=" * 70)
for tier in tier_order:
    t = df[df["customer_tier"] == tier]
    print(f"\n{'─' * 70}")
    print(f"{tier}  ({len(t):,} customers)")
    print(f"  Ever subscribed     : {t['ever_subscribed'].mean():.1%}")
    print(f"  Avg categories      : {t['n_categories_ever'].mean():.2f}")
    print(f"  Avg first-disc depth: {t['first_disc_depth'].mean():.1%}")
    print(f"  Median recency (days): {t['days_since_last_order'].median():.0f}")
    print("  First channel (%):", t["first_channel"].value_counts(normalize=True).mul(100).round(1).head(3).to_dict())

# Stacked bar: channel mix by tier
ch = df[df["customer_tier"].isin(tier_order)].copy()
ch_pct = ch.groupby(["customer_tier", "first_channel"]).size().unstack(fill_value=0)
ch_pct = ch_pct.div(ch_pct.sum(axis=1), axis=0).reindex(tier_order)
ch_pct.plot(kind="bar", stacked=True, figsize=(10, 5), colormap="Set2")
plt.title("Acquisition Channel Mix by CRM Tier")
plt.ylabel("Share of tier")
plt.xticks(rotation=15)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(OUT / "tier_channel_mix.png", dpi=150)
plt.show()


## Part I — Export decile customer table


In [ ]:
export_cols = [
    "customer_id", "true_gross_profit", "finals_revenue", "finals_orders",
    "avg_margin_pct", "cogs_coverage_pct", "profit_decile", "order_freq_decile",
    "customer_tier", "is_top_profit", "is_top_freq", "is_top_both",
    "first_channel", "first_product_cat", "ever_subscribed", "n_categories_ever",
    "first_order_date", "cohort_month", "acq_year",
]
export_df = df[[c for c in export_cols if c in df.columns]].copy()
export_df = export_df.sort_values(["profit_decile", "true_gross_profit"], ascending=[True, False])
for col in ["true_gross_profit", "finals_revenue"]:
    if col in export_df.columns:
        export_df[col] = export_df[col].round(2)

out_csv = OUT / "decile_customer_table.csv"
export_df.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")
print(f"Rows: {len(export_df):,}")
display(export_df[export_df["profit_decile"] == "D1"].head(8))


## Part J — Summary


In [ ]:
print("=" * 65)
print("SOLUTION 1 — SUMMARY")
print("=" * 65)
print(f"Decile pool (finals, excl. all-marketplace): {len(df):,}")
d1c = profit_summary[profit_summary["profit_decile"] == "D1"].iloc[0]
d1f = freq_summary[freq_summary["order_freq_decile"] == "D1"].iloc[0]
print(f"\nD1 by profit    : {d1c['n_customers']:,} customers | {d1c['pct_of_total_profit']:.1%} of profit")
print(f"D1 by frequency : {d1f['n_customers']:,} customers | {d1f['pct_of_total_orders']:.1%} of orders")
print(f"D1 overlap      : {len(d1_both):,} customers")
print("\nOutputs in:", OUT)
print("  decile_customer_table.csv")
print("  decile_profit.png | decile_order_frequency.png | decile_overlap_heatmap.png")
print("  tier_reinvestment.png | tier_channel_mix.png")
print("=" * 65)
